In [1]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import LSTM, Dense, Dropout, GlobalAveragePooling2D,Input,Bidirectional,Lambda,TimeDistributed,Conv2D,Flatten
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

2025-04-23 23:50:31.902323: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-23 23:50:31.933187: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745441431.966893   12432 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745441431.975548   12432 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-23 23:50:32.008807: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
FRAME_COUNT = 16
IMAGE_SIZE = 64
NUM_CLASSES = 3  
FOLDER_PATH = "../img/v"

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMAGE_SIZE,IMAGE_SIZE,3))
base_model.trainable = False  
feature_extractor = tf.keras.Model(inputs=base_model.input, outputs=GlobalAveragePooling2D()(base_model.output))

def load_data_transfer_learning_from_new(folder, frame_count=FRAME_COUNT, img_size=(IMAGE_SIZE,IMAGE_SIZE), num_classes=NUM_CLASSES):
    X = []
    y = []

    for video_folder in sorted(os.listdir(folder)):
        video_path = os.path.join(folder, video_folder)
        if not os.path.isdir(video_path):
            continue

        try:
            _, label = video_folder.split('_')
            label = int(label)
        except ValueError:
            print(f"Skipping {video_folder}, invalid format")
            continue

        frames = []
        frame_files = sorted(os.listdir(video_path))[:frame_count]

        for frame_file in frame_files:
            frame_path = os.path.join(video_path, frame_file)
            img = cv2.imread(frame_path)
            if img is not None:
                img = cv2.resize(img, img_size)
                img = img / 255.0
                frames.append(img)

        if len(frames) == frame_count:
            features = feature_extractor.predict(np.array(frames), verbose=0)
            X.append(features)
            y.append(label)
        else:
            print(f"Skipping {video_folder}: only {len(frames)} frames found (needs {frame_count})")

    X = np.array(X)
    y = to_categorical(np.array(y), num_classes=num_classes)

    return X, y



X, y = load_data_transfer_learning_from_new(FOLDER_PATH, frame_count=FRAME_COUNT, img_size=(IMAGE_SIZE,IMAGE_SIZE), num_classes=NUM_CLASSES)

print("X shape:", X.shape) 
print("y shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



2025-04-23 23:50:39.512334: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


X shape: (141, 16, 2048)
y shape: (141, 3)


In [3]:
print(f"Test size: {X_test.shape[0]}")
print(f"Train size: {X_train.shape[0]}")

Test size: 29
Train size: 112


In [4]:
model=Sequential([
    Input(shape=(FRAME_COUNT, 2048)),

    Bidirectional(LSTM(64, return_sequences=True)),
    Dense(64, activation='relu'),

    Bidirectional(LSTM(64, return_sequences=True)),
    Dense(64, activation='relu'),

    Bidirectional(LSTM(64, return_sequences=True)),
    Dense(64, activation='relu'),

    Bidirectional(LSTM(32, return_sequences=False)),
    Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(optimizer="RMSprop", loss='categorical_crossentropy', metrics=['accuracy'])

In [5]:
model.fit(X_train,y_train,batch_size=8,epochs=20,validation_split=0.1)

Epoch 1/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 28s 374ms/step - accuracy: 0.4552 - loss: 1.1055 - val_accuracy: 0.3333 - val_loss: 1.0879
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 159ms/step - accuracy: 0.3387 - loss: 1.0806 - val_accuracy: 0.3333 - val_loss: 1.1485
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 182ms/step - accuracy: 0.4508 - loss: 1.0688 - val_accuracy: 0.4167 - val_loss: 1.0796
Epoch 4/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 153ms/step - accuracy: 0.4033 - loss: 1.0938 - val_accuracy: 0.4167 - val_loss: 1.0838
Epoch 5/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 155ms/step - accuracy: 0.4058 - loss: 1.0935 - val_accuracy: 0.4167 - val_loss: 1.0781
Epoch 6/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 181ms/step - accuracy: 0.4433 - loss: 1.0826 - val_accuracy: 0.4167 - val_loss: 1.0818
Epoch 7/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 181ms/step - accuracy: 0.3740 - loss: 1.0855 - val_accuracy: 0.4167 - val_loss: 1.0822
Epoch 8/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 125ms/step - accuracy: 0.4702 - loss: 1.0616 - val_accuracy: 0

In [6]:
model.evaluate(X_test,y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step - accuracy: 0.3793 - loss: 1.0830


[1.0829635858535767, 0.37931033968925476]